# Spain Energy Analytics — Exploratory Analysis\n\nThis notebook explores the modeled SQLite layer produced by `scripts/run_pipeline.py`. It intentionally queries the database instead of reading raw API JSON directly, mirroring a normal analytics workflow.

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB = ROOT / "data" / "processed" / "spain_energy.db"

if not DB.exists():
    raise FileNotFoundError("Run scripts/run_pipeline.py before this notebook.")


## 1. Dataset coverage

In [ ]:
with sqlite3.connect(DB) as con:
    coverage = pd.read_sql_query(
        """
        SELECT
            i.dataset,
            COUNT(*) AS observations,
            MIN(f.timestamp_local) AS first_timestamp,
            MAX(f.timestamp_local) AS last_timestamp
        FROM fact_observation AS f
        JOIN dim_indicator AS i USING (indicator_key)
        GROUP BY i.dataset
        ORDER BY i.dataset
        """,
        con,
    )

coverage


## 2. Available indicators\nAlways inspect source indicator names and magnitudes before interpreting `value`.

In [ ]:
with sqlite3.connect(DB) as con:
    indicators = pd.read_sql_query(
        """
        SELECT dataset, indicator_title, magnitude, source_last_update
        FROM dim_indicator
        ORDER BY dataset, indicator_title
        """,
        con,
    )

indicators


## 3. Demand profile by local hour

In [ ]:
with sqlite3.connect(DB) as con:
    demand = pd.read_sql_query(
        """
        SELECT
            i.indicator_title,
            f.hour_local,
            AVG(f.value) AS average_value
        FROM fact_observation AS f
        JOIN dim_indicator AS i USING (indicator_key)
        WHERE i.dataset = 'demand'
        GROUP BY i.indicator_title, f.hour_local
        ORDER BY i.indicator_title, f.hour_local
        """,
        con,
    )

demand.head()


In [ ]:
if not demand.empty:
    pivot = demand.pivot(
        index="hour_local",
        columns="indicator_title",
        values="average_value",
    )
    ax = pivot.plot(figsize=(10, 5))
    ax.set_title("Average demand profile by local hour")
    ax.set_xlabel("Local hour")
    ax.set_ylabel("Source value — inspect magnitude")
    plt.tight_layout()


## 4. Generation indicators

In [ ]:
with sqlite3.connect(DB) as con:
    generation = pd.read_sql_query(
        """
        SELECT
            i.indicator_title,
            i.magnitude,
            AVG(f.value) AS average_value
        FROM fact_observation AS f
        JOIN dim_indicator AS i USING (indicator_key)
        WHERE i.dataset = 'generation'
        GROUP BY i.indicator_title, i.magnitude
        ORDER BY average_value DESC
        """,
        con,
    )

generation.head(15)


## 5. Price extremes

In [ ]:
with sqlite3.connect(DB) as con:
    price_extremes = pd.read_sql_query(
        """
        SELECT
            i.indicator_title,
            f.timestamp_local,
            f.value,
            i.magnitude
        FROM fact_observation AS f
        JOIN dim_indicator AS i USING (indicator_key)
        WHERE i.dataset = 'price'
        ORDER BY f.value DESC
        LIMIT 20
        """,
        con,
    )

price_extremes
